# SBI Tutorial: Detect the Higgs boson!

In this tutorial, you will get acquainted—and play—with the central concepts of simulation-based inference: the simulator and the inference network, while solving a—hopefully—seemingly simple problem (which may prove challenging using traditional, lkikelihood-based, techniques: you are invited to attempt a "traditional" solution to compare).

Concretely, you will discover (in mock data, which you will produce yourself; you are invited to find and analyse the real data, if interested) the Higgs boson! To do that, consider the following forward model:
- We perform an experiment, which generates $R_s$ Higgs bosons per unit time. This means that, for an "integration" period $T$, the true number of "signal events" $N_s$ will be Poisson distributed:
  $$N_s \sim \mathrm{Poisson}(TR_s).$$
  From theory, we expect the rate to be
  $$R_s \sim \mathrm{Uniform}(0, 100)$$
  per suitable units of $T$ (assume $T=1$ for the time being). What would $R_s = 0$ mean?
- The apparatus also detects *background* events from various sources of noise. Assuming their rate is
  $$R_b \sim \mathrm{Uniform}(5000, 15000),$$
  the corresponding number of detections is
  $$N_b \sim \mathrm{Poisson}(TR_b).$$
- The total number of detections is $N \equiv N_s + N_b$. At this point it is worth it to pause and think: can we (how?) "detect" the Higgs boson, based solely on the number of detected events?

- Besides simply detecting particles, detectors can measure their energies / masses. Suppose the Higgs boson's mass is $m_H$. From *prior* considerations, we expect:
  $$m_H \sim \mathrm{Uniform}(100, 150) \,\mathrm{GeV}.$$
- For every detected Higgs boson $i$, we record a noisy measurement of $m_H$:
  $$E_i \sim \mathrm{Normal}(m_H, \sigma^2),$$
  where $\sigma$ is the standard deviation of the mass/energy measurement error/uncertainty (be careful in the implementation: most frameworks parametrize using $\sigma$, but in literature it's usual to use $\sigma^2$ when writing the normal distribution). Our aparatus is not perfectly calibrated, so
  $$\sigma \sim \mathrm{Uniform}(1, 10) \,\mathrm{GeV}.$$
- Since background events come from many sources, the distribution of their (measured, i.e. after instrumental noise) energies can be effectively modelled as a decaying exponential:
  $$E_j \sim \mathrm{Exponential}(E_b),$$
  where $E_b$ is the characteristic (average) energy of background events. (Be careful in the implementation, since often the Exponential is described in terms of a rate parameter, e.g. $\lambda_b \equiv 1/E_b$.)
  The exact properties of the noise are uncertain a priori:
  $$E_b \sim \mathrm{Uniform}(10, 50) \,\mathrm{GeV}.$$

- The final data set is composed of an unordered set containing the energies of all detected events, without the information whether each is "source" or "background".


## Tasks
1. Implement this model as a forward simulator. It has to be a function that samples from all mentioned distributions and returns all sampled values, including those for the global parameters (which are they? are there any latent/per-"object" parameters?) and the final data set, as a dictionary with appropriate keys.

   1.1. Think of a way to "summarize" or represent the final data in a standard form (and possibly implement it in the simulator or separate function). This represents the "data reduction" stage of an analysis, which is also part of the forward model.

2. Generate a mock data set. Can you somehow set the parameter values at will?[^pp] (What does that tell you, philosophically, about SBI methods?) If not, rewrite your simulator, so that you can, and set
   $$
   m_H = 126\,\mathrm{GeV},
   \quad R_s=40,
   \quad R_b = 10000,
   \quad E_b = 20\,\mathrm{GeV},
   \quad \sigma = 1.7\,\mathrm{GeV}$$.
   For the rest of your life, pretend this is the real data and deny the existence of the outside world.

   2.1. Viusualize your data in a suitable way. Can you detect the Higgs boson by eye? What parameters can you change, to make the signal stronger or weaker (by eye)? Play around with parameter values!

3. Identify the parameters of interest (to someone studying the Higgs boson). Perform your favourite flavour of SBI to infer them from the data you just generated.

    3.1. How does the data representation affect the architecture of the neural network you will / would like to use?
    3.2. Run validation tests on the inference. Try to spot patterns regarding the precision of the posteriors.
   
4. Perform model selection using this model vs. the alternative no-Higgs-boson model (find a suitable parameter you can set to zero to realise this model).

    4.1. Explore the dependence of the Bayes factor on various relevant parameters (e.g. controlling the noise).

Bonus / advanced tasks:

1. Right now, the role of $T$ is of a fixed *simulator/model setting*. Can you relax this role? First, think how you expect a longer integration time to affect the results, especially of the model comparison. Re-formulate the simulator and inference so that $T$ will be part of the *data*, i.e. it is an a-priori unknown ($T \sim \mathrm{Uniform}(1, 10)$) but finally observed random variable. Input it into the networks alongside the counts/energies and explore the results *as a function* of $T$.
2. Modify the simulator, so that some low-energy events are missed (say, the probability of detection is 0 for energies below $100\,\mathrm{GeV}$ and rises linearly to 1 at $150\,\mathrm{GeV}$). "Validate" your previously trained inference network using these new simulations.


[^pp]: If you used a probabilistic programming language like [Pyro](https://pyro.ai/), maybe yes!

In [ ]:
import torch
from torch.distributions import Exponential, Normal, Poisson, Uniform
from matplotlib import pyplot as plt

### Simulator

In [ ]:
def simulate(params=None):
    if params is None:
        params = {}

    settings:
    T = 1

    # Sample values from appropriate distributions...

    # Higgs bosons:
    Rs = ...
    Ns = ...
    mH = ...
    sigma = ...
    Ei = ...  # Ns values

    # background:
    Rb = ...
    Nb = ...
    Eb = ...
    Ej = ...  # Nb values

    # combine and shuffle all events
    data = ...

    # Can you think of a useful "standardized" representation of the data?
    # If yes, you can calculate it here and add it to the list of returned variables below.

    return {key: locals().get(key) for key in (
        # Return the following local variables:
        'Rs', 'Ns', 'mH', 'sigma', 'Ei',
        'Rb', 'Nb', 'Eb', 'Ej',
        'data'
    )}

In [ ]:
data = simulate(params={'mH': 126, 'Rs': 40, 'Rb': 10000, 'Eb': 20, 'sigma': 1.7})

#### Plot and play around with the data

In [ ]:
# plt. ...

### Inference

In [ ]:
# A list of parameters we want to learn (jointly)
PARAM_NAMES = 'mH', 'Rs'

# A list of variables we consider "observed": e.g. ['data']
# (but consider that this has to be pluggable in your neural network afterwards.
OBS_NAMES = 'hist',

In [ ]:
# Helper functions to extract the relevant variables from raw simulations

from tqdm import trange

def sim_to_theta(s, param_names):
    return torch.cat([torch.atleast_1d(s[key]) for key in param_names], -1)

def sim_to_d(s, obs_names):
    return torch.cat([torch.atleast_1d(s[key]) for key in obs_names], -1)

def simulate_dataset(n, param_names, obs_names):
    sims = [
        {key: s[key] for key in (*param_names, *obs_names)}
        for _ in trange(n) for s in [simulate()]
    ]
    theta = torch.stack([sim_to_theta(s, param_names) for s in sims], 0)
    x = torch.stack([sim_to_d(s, obs_names) for s in sims], 0)

    return theta, x

In [ ]:
# Simulate training data. Maybe you also need some validation examples?
theta, d = simulate_dataset(10000, PARAM_NAMES, OBS_NAMES)

#### Easy posterior estimation

It's super easy with the [SBI framework](https://sbi.readthedocs.io/en/latest/)!
I'll show basic inference and leave the rest of the tasks to you. The documentation of the SBI package is a great resource that should help you with them.

However, if you'd like to really understand what's going on, I'd suggest implementing the methods yourself.

In [ ]:
from sbi.inference import NPE
from sbi.analysis import pairplot

npe = NPE(prior=None).append_simulations(theta, d)
density_estimator = npe.train()
post = npe.build_posterior()

In [ ]:
samples = post.sample((1000,), x=sim_to_d(data, OBS_NAMES))
pairplot(samples, labels=[{
    'Rs': r'$R_s$', 'Ns': r'$N_s$',
    'mH': r'$m_H$', 'sigma': r'$\sigma$',
    'Rb': r'$R_b$', 'Nb': r'$N_b$',
    'Eb': r'$E_b$',
}[key] for key in PARAM_NAMES]);

#### Hard posterior estimation (optional)

Implement a posterior estimator yourself!
Hint: You can start with a simple 1D problem, inferring only $m_H$. You network needs to predict the posterior mean and standard deviation, i.e. just two numbers. You can then easily compute the loss and write a simple training / optimization loop (the normalization will be imposed by you using the correctly normalized Gaussian PDF).

### Model comparison

Generate two data sets (one with parameters sampled from their priors, one where $R_s=0$ in each simulation, but all other parameters are free). Then implement a simple binary classifier: your network just needs to output one number (or two, which you then normalize), and you can use PyTorch's built-in [binary cross-entropy loss](https://docs.pytorch.org/docs/2.13/generated/torch.nn.BCELoss.html) (or better, write the loss yourself!).